# OCR de Kanjis N1 com YOLOv8
Este notebook foi feito para ser rodado no **Kaggle** ou **Google Colab** (com GPU ativada!).

Ele funciona orquestrando os códigos profissionais modulares que você já tem no seu GitHub.

## 1. Instalar as Dependências e Baixar o Repositório

In [ ]:
!pip install ultralytics

# Previne erro de tentar clonar numa pasta que já existe se você rodar a célula duas vezes
!rm -rf OCR-de-kanjis-N1-com-YoloV8
!git clone https://github.com/MiguelMussalam/OCR-de-kanjis-N1-com-YoloV8.git
%cd OCR-de-kanjis-N1-com-YoloV8

## 2. Gerar o Dataset Sintético
Vamos rodar os seus scripts locais para baixar os dados, baixar as fontes e gerar as imagens.

In [ ]:
!python src/data/dowload_fonts.py
!python src/data/download_n1_kanjis.py
!python src/data/generate_synthetic_images.py

## 3. Treinar o Modelo YOLOv8
Agora iniciamos o treinamento utilizando a base sintética que acabamos de gerar!
Note o caminho do `data.yaml` que foi criado no passo anterior.

In [ ]:
import os
from ultralytics import YOLO

model = YOLO('yolov8n.pt') # Modelo Nano (leve e rápido). Você pode tentar yolov8s.pt depois.

# Verificando o caminho real do arquivo data.yaml
data_path = os.path.abspath("data/synthetic/data.yaml")

results = model.train(
    data=data_path,
    epochs=50,       # Ajuste conforme precisar (comece com 50-100)
    imgsz=640,
    batch=16,
    device=0,        # Diz para usar a placa de vídeo (GPU)
    plots=True
)

## 4. Baixar os Pesos Treinados
Ao fim do treinamento, os melhores pesos do seu modelo vão ficar salvos no diretório descrito abaixo.
A célula gera um link para você clicar e fazer o download do seu modelo compilado!

In [ ]:
from IPython.display import FileLink

# O caminho padrão do ultralytics costuma ser runs/detect/train/weights/best.pt
import glob
import os

train_dirs = glob.glob('runs/detect/train*')
if train_dirs:
    latest_train = max(train_dirs, key=os.path.getmtime)
    weights_path = f"{latest_train}/weights/best.pt"

    print(f"Seus pesos perfeitos estão em: {weights_path}")
    display(FileLink(weights_path))
else:
    print("Nenhum modelo treinado encontrado ainda.")


## 5. Testar o Modelo (Upload Manual do Manga109)
Já que você mencionou que fará o upload manual do Manga109 para testar na nuvem, a célula abaixo cuida de tudo.

1. Ela criará uma pasta chamada `manga_test`.
2. Ela pedirá para você ir no painel do Kaggle/Colab e fazer o **upload das imagens** do Manga109 diretamente para dentro dessa pasta.
3. Depois disso, rodando a célula de novo, ela pegará automaticamente o arquivo ótimo de pesos que acabou de ser gerado, rodará a inferência sobre todas as imagens upadas, exibirá o primeiro resultado na tela e gerará um arquivo `.zip` para você baixar contendo todas as predições de uma vez!

In [ ]:
from ultralytics import YOLO
import glob
import os
import matplotlib.pyplot as plt
import cv2
from IPython.display import FileLink, display

# 1. Definir a pasta para onde você vai jogar as imagens
test_images_folder = 'manga_test/'
os.makedirs(test_images_folder, exist_ok=True)

print(f"⭐ ATENÇÃO: Acesse a área de pastas e faça o upload de imagens de teste dentro da pasta '{test_images_folder}'!")

imagens_upadas = glob.glob(f'{test_images_folder}/*.jpg') + glob.glob(f'{test_images_folder}/*.png')

if not imagens_upadas:
    print("\n⚠️ Nenhuma imagem encontrada ainda...")
    print("Vá no menu de arquivos, abra a pasta 'manga_test' e faça o upload.")
    print("Depois de feita a transferência, MANTENHA A CALMA e rode esta célula de novo :)")
else:
    print(f"\n✅ {len(imagens_upadas)} imagens encontradas para detecção!")
    
    # 2. Carregar o modelo treinado perfeito
    train_dirs = glob.glob('runs/detect/train*')
    latest_train = max(train_dirs, key=os.path.getmtime)
    weights_path = f"{latest_train}/weights/best.pt"
    
    print(f"Carregando modelo em: {weights_path}...")
    model = YOLO(weights_path)
    
    # 3. Rodar inferência nas imagens da pasta manga_test
    print("\nRodando as predições...")
    results = model(test_images_folder, conf=0.25)
    
    # 4. Exibir um exemplo
    predict_dirs = glob.glob('runs/detect/predict*')
    if predict_dirs:
        latest_pred = max(predict_dirs, key=os.path.getmtime)
        imgs = glob.glob(f'{latest_pred}/*.jpg')
        
        if imgs:
            print(f"\n🖼️ Exibindo resultado de: {imgs[0]}")
            img = cv2.imread(imgs[0])
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            plt.figure(figsize=(10, 10))
            plt.imshow(img)
            plt.axis('off')
            plt.show()
            
            # 5. Zipar todos os resultados detectados
            print("\nEmpacotando os resultados...")
            cmd = f'zip -q -r resultados_manga.zip {latest_pred}'
            os.system(cmd)
            print("Baixe todas as imagens clicando abaixo:")
            display(FileLink('resultados_manga.zip'))
